# RSNA Knee — Soumission (vision only)

Notebook **self-contained** pour Kaggle (internet off) :
DICOM test → volumes (24, 224, 224) → KneeNet (ConvNeXt-tiny 2.5D, 3 plans) → `submission.csv`.

**Avant de soumettre** : attacher un dataset Kaggle contenant `best.pt`
(poids entraînés en local) et ajuster `WEIGHTS_PATH` ci-dessous.

Code synchronisé avec `src/infer/` et `src/models/kneenet.py` du repo local.

In [ ]:
import os
from pathlib import Path

# --- Configuration ---------------------------------------------------------
KAGGLE = os.path.exists('/kaggle')
if KAGGLE:
    DATA_DIR = Path('/kaggle/input')
    print('contenu /kaggle/input :', sorted(p.name for p in DATA_DIR.iterdir()))
    # Kaggle monte /kaggle/input/competitions/<comp> et /kaggle/input/datasets/<ds>
    # -> ne PAS rglob (500k DICOM = tres lent) : on liste les sous-dossiers directs.
    COMP_DIR = None
    for base in [DATA_DIR / 'competitions', DATA_DIR]:
        if base.exists():
            for p in sorted(base.iterdir()):
                if (p / 'test.csv').exists():
                    COMP_DIR = p
                    break
        if COMP_DIR is not None:
            break
    if COMP_DIR is None:
        raise FileNotFoundError('test.csv introuvable sous /kaggle/input')
    # poids : datasets/<ds>/best.pt (datasets = peu de fichiers, rglob ok)
    _w = list((DATA_DIR / 'datasets').rglob('best.pt')) if (DATA_DIR / 'datasets').exists() else []
    if not _w:
        _w = list(DATA_DIR.rglob('best.pt'))
    if not _w:
        raise FileNotFoundError('best.pt introuvable (dataset weights attache ?)')
    WEIGHTS_PATH = _w[0]
else:
    COMP_DIR = Path('../data/csv')          # debug local (CSV seulement)
    WEIGHTS_PATH = Path('../checkpoints/baseline_v5_nopw/best.pt')

TEST_CSV = COMP_DIR / 'test.csv'
TEST_SERIES_CSV = COMP_DIR / 'test_series.csv'
TEST_SERIES_DIR = COMP_DIR / 'test_series'
SAMPLE_SUB = COMP_DIR / 'sample_submission.csv'

# Device : essai CUDA, repli CPU si le GPU n'est pas supporte par l'image
# Kaggle ('no kernel image available'). Le pretraitement DICOM domine le
# temps, donc un repli CPU reste largement dans la limite des 9 h.
import torch

def _pick_device():
    if not torch.cuda.is_available():
        return 'cpu'
    try:
        m = torch.nn.Conv2d(3, 8, 3, padding=1).cuda()
        _ = m(torch.zeros(1, 3, 32, 32, device='cuda')).sum().item()
        return 'cuda'
    except Exception as e:
        print('CUDA inutilisable, repli CPU :', str(e)[:120])
        return 'cpu'

DEVICE = _pick_device()
AMP_DTYPE = (torch.bfloat16
             if (DEVICE == 'cuda' and torch.cuda.is_bf16_supported())
             else torch.float16)
DEBUG_COMPARE_NPZ = True  # compare nos volumes aux stats NPZ train

# moments des NPZ train (uint8) mesures en local sur 400 volumes
NPZ_MEAN_U8, NPZ_STD_U8 = 42.0, 49.5

print('competition dir :', COMP_DIR)
print('poids           :', WEIGHTS_PATH, '| existe :', WEIGHTS_PATH.exists())
print('device          :', DEVICE)

In [ ]:
# --- Pretraitement DICOM -> volume (24, 224, 224) uint8 --------------------
# (copie self-contained de src/infer/preprocess_dicom.py)
import cv2
import numpy as np
import pydicom

N_SLICES, IMG_SIZE = 24, 224


def _slice_sort_key(ds):
    inst = getattr(ds, 'InstanceNumber', None)
    if inst is not None:
        try:
            return float(inst)
        except (TypeError, ValueError):
            pass
    ipp = getattr(ds, 'ImagePositionPatient', None)
    iop = getattr(ds, 'ImageOrientationPatient', None)
    if ipp is not None and iop is not None:
        row = np.array(iop[:3], dtype=np.float64)
        col = np.array(iop[3:], dtype=np.float64)
        normal = np.cross(row, col)
        return float(np.dot(np.array(ipp, dtype=np.float64), normal))
    return float(ipp[2]) if ipp is not None else 0.0


def load_series_dicom(series_dir):
    files = sorted(Path(series_dir).glob('*.dcm'))
    datasets = []
    for f in files:
        try:
            datasets.append(pydicom.dcmread(str(f)))
        except Exception:
            continue
    if not datasets:
        raise RuntimeError(f'aucun DICOM lisible dans {series_dir}')
    datasets.sort(key=_slice_sort_key)

    slices = []
    for ds in datasets:
        arr = ds.pixel_array.astype(np.float32)
        arr = arr * float(getattr(ds, 'RescaleSlope', 1.0) or 1.0) \
                  + float(getattr(ds, 'RescaleIntercept', 0.0) or 0.0)
        if getattr(ds, 'PhotometricInterpretation', 'MONOCHROME2') == 'MONOCHROME1':
            arr = arr.max() - arr
        slices.append(cv2.resize(arr, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA))

    vol = np.stack(slices)
    lo, hi = np.percentile(vol, 1.0), np.percentile(vol, 99.0)
    if hi <= lo:
        hi = lo + 1.0
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)
    idx = np.linspace(0, vol.shape[0] - 1, N_SLICES).round().astype(int)
    return (vol[idx] * 255.0).round().astype(np.uint8)


def study_to_tensor(study_dir, series_rows):
    """(3, 24, 224, 224) float32 dans [0, 1] ; plan absent -> zeros."""
    out = np.zeros((3, N_SLICES, IMG_SIZE, IMG_SIZE), dtype=np.float32)
    for i, plane in enumerate(['Sagittal', 'Coronal', 'Axial']):
        rows = series_rows[series_rows.Anatomical_Plane == plane]
        if len(rows) == 0:
            continue
        # Fluid_Sensitive et Fat_Suppression restent deux colonnes distinctes
        fluid = rows[rows.Fluid_Sensitive == 1] if 'Fluid_Sensitive' in rows else rows.iloc[0:0]
        if len(fluid) == 0 and 'Fat_Suppression' in rows:
            fluid = rows[rows.Fat_Suppression == 1]
        pick = (fluid if len(fluid) else rows).sort_values('SeriesInstanceUID').iloc[0]
        sdir = Path(study_dir) / pick.SeriesInstanceUID
        if not sdir.exists():
            continue
        try:
            out[i] = load_series_dicom(sdir).astype(np.float32) / 255.0
        except Exception as e:
            print(f'  [warn] {pick.SeriesInstanceUID}: {e}')
    return out

In [ ]:
# --- Modele (copie self-contained de src/models/kneenet.py) -----------------
import timm
import torch
import torch.nn as nn

TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
           'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's",
           'Contusion', 'Fracture']
NORM_MEAN, NORM_STD = 0.1648, 0.1941


class KneeNet(nn.Module):
    def __init__(self, backbone='convnext_tiny', n_series=3, head_dim=512, dropout=0.2):
        super().__init__()
        self.n_series = n_series
        self.encoder = timm.create_model(backbone, pretrained=False, in_chans=24,
                                         num_classes=0, global_pool='avg')
        f = self.encoder.num_features
        self.fusion = nn.Sequential(
            nn.LayerNorm(n_series * f),
            nn.Linear(n_series * f, head_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_dim, len(TARGETS)),
        )

    def forward(self, x):
        b, s, d, h, w = x.shape
        feats = self.encoder(x.reshape(b * s, d, h, w))
        return self.fusion(feats.reshape(b, s * feats.shape[-1]))


ckpt = torch.load(str(WEIGHTS_PATH), map_location=DEVICE, weights_only=False)
cfg = ckpt.get('config', {})
model = KneeNet(backbone=cfg.get('backbone', 'convnext_tiny'),
                head_dim=cfg.get('head_dim', 512),
                dropout=cfg.get('dropout', 0.2))
model.load_state_dict(ckpt['model'])
model.to(DEVICE).eval()
print('modele charge | val macro-AUC local :', round(ckpt.get('val_macro_auc', float('nan')), 4))

In [ ]:
# --- Validation du pretraitement : nos volumes DICOM vs stats NPZ ----------
# Au scoring, train_series/ est remplace par test_series/ ; en debug Kaggle,
# train_series/ est disponible : on compare les moments a ceux des NPZ locaux.
import pandas as pd

if DEBUG_COMPARE_NPZ:
    ref_dir = COMP_DIR / 'train_series'
    ref_csv = COMP_DIR / 'train_series.csv'
    if ref_dir.exists() and ref_csv.exists():
        ref = pd.read_csv(ref_csv)
        sample = ref.sample(min(5, len(ref)), random_state=0)
        for _, r in sample.iterrows():
            sdir = ref_dir / r.StudyInstanceUID / r.SeriesInstanceUID
            try:
                v = load_series_dicom(sdir)
                print(f'{r.SeriesInstanceUID[:40]}... shape {v.shape} '
                      f'mean {v.mean():.1f} (NPZ~{NPZ_MEAN_U8}) std {v.std():.1f} (NPZ~{NPZ_STD_U8})')
            except Exception as e:
                print('  [warn]', e)
    else:
        print('train_series/ absent (run de scoring) — comparaison ignoree')

In [ ]:
# --- Inference sur le test --------------------------------------------------
import time

test = pd.read_csv(TEST_CSV)
test_series = pd.read_csv(TEST_SERIES_CSV)
print(f'{len(test)} etudes a predire')

rows = []
t0 = time.time()
for i, uid in enumerate(test.StudyInstanceUID):
    vol = study_to_tensor(TEST_SERIES_DIR / uid,
                          test_series[test_series.StudyInstanceUID == uid])
    x = torch.from_numpy(vol).unsqueeze(0).to(DEVICE)
    x = (x - NORM_MEAN) / NORM_STD
    # fp32 : AMP fp16 + ConvNeXt 24 canaux echoue sur T4
    # ('GET was unable to find an engine to execute this computation')
    with torch.no_grad():
        try:
            logits = model(x)
        except RuntimeError as e:
            print('forward CUDA echoue, cudnn off puis CPU :', str(e)[:80])
            torch.backends.cudnn.enabled = False
            try:
                logits = model(x)
            except RuntimeError:
                DEVICE = 'cpu'
                model = model.to('cpu')
                x = x.cpu()
                logits = model(x)
    prob = torch.sigmoid(logits.float()).cpu().numpy()[0]
    rows.append([uid] + prob.tolist())
    if (i + 1) % 50 == 0:
        print(f'{i+1}/{len(test)} | {(time.time()-t0)/60:.1f} min', flush=True)

pred = pd.DataFrame(rows, columns=['StudyInstanceUID'] + TARGETS)
pred.head()

In [ ]:
# --- submission.csv au format exact de sample_submission --------------------
sample = pd.read_csv(SAMPLE_SUB)
sub = pred.set_index('StudyInstanceUID').loc[sample.StudyInstanceUID].reset_index()
sub = sub[list(sample.columns)]
sub.to_csv('submission.csv', index=False)

print(sub.shape, '| colonnes OK :', list(sub.columns) == list(sample.columns))
print(sub[TARGETS].describe().loc[['mean', 'std', 'min', 'max']].round(3).to_string())
assert not sub[TARGETS].isna().any().any(), 'NaN dans la submission !'
sub.head()